# Supply Chain Control Tower — Laya Triage (Colab Notebook)

**Manufacturing control tower** prototype using [Laya](https://github.com/) typed decisions (`choice`, `score`, `noul`) — not free-form LLM text.

> **Research / decision-support only.** Calibrate probabilities and keep humans in the loop before operational procurement automation.

## What this notebook does

1. Mount Google Drive and use the `supply_chain_risk_laya` project folder
2. Install dependencies and load the Laya `Router`
3. Run the **port strike** demo alert
4. Triage the **10-alert sample library** (`sample_alerts.json`)
5. *(Optional)* Batch triage on **AG News** headlines
6. Write `output/control_tower_results.json` + `.csv` and preview for Power BI / Fabric


## 1. Google Drive & project path

Edit `PROJECT_DIR` if your folder is not under `MyDrive/supply_chain_risk_laya`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# --- edit if your Drive layout differs ---
PROJECT_DIR = Path('/content/drive/MyDrive/supply_chain_risk_laya')
os.environ['SCM_PROJECT_ROOT'] = str(PROJECT_DIR)

assert PROJECT_DIR.is_dir(), f"Missing folder: {PROJECT_DIR}. Upload the project from GitHub/Drive first."
%cd $SCM_PROJECT_ROOT
print('Project root:', PROJECT_DIR)


## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

import laya
print('laya version:', getattr(laya, '__version__', 'unknown'))


## 3. Imports & configuration

Thresholds and paths come from `config.py` (overridable via environment variables).

In [ ]:
import sys
import time
import json
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path(os.environ['SCM_PROJECT_ROOT'])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import config
import supply_chain_risk_laya as scr

scr.configure_logging()

print('Backup sourcing threshold:', config.BACKUP_SOURCING_THRESHOLD)
print('Relevance threshold:', config.RELEVANCE_THRESHOLD)
print('Output dir:', config.OUTPUT_DIR)


## 4. Initialize Laya Router

First run may download Hugging Face checkpoints (large). Use `preload=False` on low-memory runtimes.

In [ ]:
PRELOAD = True  # set False to reduce RAM (slower first inference)

t0 = time.perf_counter()
router = scr.create_router(preload=PRELOAD)
init_sec = time.perf_counter() - t0
print(f'Initialization time: {init_sec:.2f} sec')
print('Device:', scr.detect_compute_device())


## 5. Primary demo — port strike alert

Same scenario as `python supply_chain_risk_laya.py` default demo.

In [ ]:
t0 = time.perf_counter()
demo_result = scr.run_demo_port_strike(router)
infer_sec = time.perf_counter() - t0
print(f'Inference time: {infer_sec:.3f} sec\n')
scr.print_control_tower_report(demo_result)


## 6. Try your own alert

Edit `custom_alert` below and re-run the cell.

In [ ]:
custom_alert = {
    'source_type': 'Macro News Alert',
    'alert_text': (
        'Customs officials closed a border crossing for 72 hours after a security incident, '
        'blocking truck shipments of industrial polymers to two manufacturing sites.'
    ),
}

custom_result = scr.analyze_supply_chain_risk(custom_alert, router=router)
scr.print_control_tower_report(custom_result)


## 7. Sample alert library (10 records)

In [ ]:
library_results = []
for alert in scr.load_sample_alerts():
    parsed = scr.analyze_supply_chain_risk(alert, router=router)
    parsed['id'] = alert.get('id')
    library_results.append(parsed)

rows = [scr.flatten_for_csv(r) for r in library_results]
library_df = pd.DataFrame(rows)
library_df.insert(0, 'id', [r.get('id') for r in library_results])
library_df


## 8. Optional — AG News batch

**Limitation:** AG News is general news classification, not supply-chain ground truth. Use for unstructured text demos only.

Set `RUN_AG_NEWS = True` to enable (adds several minutes on CPU).

In [ ]:
RUN_AG_NEWS = False
AG_NEWS_LIMIT = config.AG_NEWS_SAMPLE_SIZE

ag_news_df = None
if RUN_AG_NEWS:
    dataset = scr.load_ag_news_dataset()
    scr.inspect_ag_news_dataset(dataset)
    t0 = time.perf_counter()
    ag_news_df = scr.process_ag_news(dataset, router, limit=AG_NEWS_LIMIT)
    batch_sec = time.perf_counter() - t0
    print(f'AG News batch: {batch_sec:.2f} sec for {len(ag_news_df)} rows')
    display(ag_news_df.head(10))
else:
    print('AG News batch skipped (RUN_AG_NEWS=False)')


## 9. Combine results & save outputs

In [ ]:
all_results = [demo_result, custom_result] + library_results
if ag_news_df is not None and len(ag_news_df):
    for record in ag_news_df.to_dict(orient='records'):
        all_results.append(record)

json_path, csv_path = scr.save_results(all_results)
print('Wrote:', json_path)
print('Wrote:', csv_path)

preview_df = pd.read_csv(csv_path)
preview_df


## 10. Risk summary charts (optional)

In [ ]:
if ag_news_df is not None and len(ag_news_df):
    scr.plot_risk_summary(ag_news_df)
else:
    # Chart library_df categories / severity
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    library_df['disruption_category'].value_counts().plot(kind='bar', ax=ax[0], title='Sample library — category')
    library_df['severity_index'].value_counts().sort_index().plot(kind='bar', ax=ax[1], title='Sample library — severity')
    plt.tight_layout()
    plt.show()


## 11. Business Central payload preview (no API call)

In [ ]:
bc_payload = scr.export_for_business_central(all_results)
print(f'Records prepared: {len(bc_payload)}')
sample = json.dumps(bc_payload[0], indent=2)
print(sample[:2000] + ('\n...' if len(sample) > 2000 else ''))


## 12. Quick schema validation (unit tests)

Runs deterministic tests without downloading extra model weights (`not integration`).

In [ ]:
!pytest tests/test_risk_schema.py -m "not integration" -q


---

### Next steps

| Phase | Action |
|-------|--------|
| Domain data | Label your own supply-chain alerts |
| Calibration | Validate Laya probabilities before automation |
| ERP | Wire `export_for_business_central()` to Business Central |
| Analytics | Ingest CSV into Fabric / Power BI Control Tower |

CLI equivalent: `python supply_chain_risk_laya.py --sample-library --ag-news --plot`
